In [1]:
import torch
import matplotlib.pyplot as plt

from metarcwa import Model, PlaneWave, Stack, Lattice, IsotropicMedium
from metarcwa.model.utils import from_metashapes, from_dispertorch
from metarcwa.solver.config import Config
from metarcwa.solver.base import Solver

In [2]:
from dispertorch import material, list_materials, ConstantEps

# Create input and output media
incidence = IsotropicMedium(from_dispertorch(ConstantEps(1.0)))
transmission = IsotropicMedium(from_dispertorch(material('aSi')))

# Create Lattice
Lx = 400
Ly = 400
lattice = Lattice.rectangular(Lx,Ly)

# Create Stack
stack = Stack(incidence, layers=[], transmission=transmission, lattice=lattice)

# Create Source
wl = torch.linspace(400,800,50)
theta = torch.linspace(0, 45, 46)
phi = torch.linspace(0, 10, 11)
s_amp = 1.0
p_amp = 0.0

source = PlaneWave(wl, s_amp, p_amp, theta, phi)

# Create model
model = Model(stack, source)

In [15]:
# Create config
config = Config(dtype = torch.float32, device="cuda:0", 
                nx = 32, ny = 32, m = 1, n = 1, factorization=None, truncation="rectangular")

# Create solver
solver = Solver(model, config)

In [16]:
S = solver.solve()
S11, S12, S21, S22 = S.a, S.b, S.c, S.d

In [17]:
S11.shape

(torch.Size([50, 46, 11, 9, 9]),
 torch.Size([50, 46, 11, 9, 9]),
 torch.Size([50, 46, 11, 9, 9]),
 torch.Size([50, 46, 11, 9, 9]))